## Setup and Imports

In [7]:
print(sys.path)

['c:\\Users\\alexa\\anaconda3\\envs\\where-the-hull-are-you\\python313.zip', 'c:\\Users\\alexa\\anaconda3\\envs\\where-the-hull-are-you\\DLLs', 'c:\\Users\\alexa\\anaconda3\\envs\\where-the-hull-are-you\\Lib', 'c:\\Users\\alexa\\anaconda3\\envs\\where-the-hull-are-you', '', 'c:\\Users\\alexa\\anaconda3\\envs\\where-the-hull-are-you\\Lib\\site-packages', 'C:\\Users\\alexa\\git\\where-the-hull-are-you\\shared-tracking-metrics\\src', 'c:\\Users\\alexa\\anaconda3\\envs\\where-the-hull-are-you\\Lib\\site-packages\\win32', 'c:\\Users\\alexa\\anaconda3\\envs\\where-the-hull-are-you\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\alexa\\anaconda3\\envs\\where-the-hull-are-you\\Lib\\site-packages\\Pythonwin', 'c:\\Users\\alexa\\git\\where-the-hull-are-you\\model-training\\src', 'c:\\Users\\alexa\\git\\where-the-hull-are-you\\model-training\\src', 'c:\\Users\\alexa\\git\\where-the-hull-are-you\\model-training\\src', 'c:\\Users\\alexa\\git\\where-the-hull-are-you\\model-training\\src']


In [ ]:
from pathlib import Path
import pandas as pd
import sys
from loguru import logger

sys.path.append(str(Path.cwd().parent / "src"))

from evaluation.labeled_evaluator import LabeledEvaluator
from evaluation.unlabeled_evaluator import UnlabeledEvaluator
from config.model_settings import MODEL_PATH

## Load Configuration

In [ ]:
# eval_config = TrainingConfig(Path("../config/evaluation_config.yaml"))

# # Paths
# LABELED_DATA_YAML = Path(eval_config.get('data.labeled_dataset_yaml'))
# UNLABELED_VIDEO_DIR = Path(eval_config.get('data.unlabeled_videos_dir'))
# OUTPUT_DIR = Path(eval_config.get('output.metrics_dir'))

# # Models to compare
# MODELS = {
#     "YOLOv8n (baseline)": eval_config.get('model.baseline_model'),
#     "Custom trained": Path(eval_config.get('model.custom_model')),
# }

# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Unlabeled Test Videos

In [ ]:
# video_loader = VideoLoader(UNLABELED_VIDEO_DIR)
# test_videos = video_loader.get_all_videos()

# print(f"Found {len(test_videos)} test videos:")
# for video in test_videos:
#     print(f"  - {video.name}")

## Evaluate on Labeled Data

In [ ]:
# labeled_results = {}

# for model_name, model_path in MODELS.items():
#     print(f"\nEvaluating {model_name} on labeled data...")
    
#     evaluator = LabeledEvaluator(model_path)
#     evaluator.load_model()
    
#     metrics = evaluator.evaluate(LABELED_DATA_YAML)
#     labeled_results[model_name] = metrics
    
#     print(f"Results:")
#     for metric_name, value in metrics.items():
#         print(f"  {metric_name}: {value:.4f}")

## Evaluate on Unlabeled Data

In [ ]:
unlabeled_results = {}


logger.info(f"\nEvaluating {model_name} on unlabeled videos...")
    
evaluator = UnlabeledEvaluator(MODEL_PATH)
evaluator.load_model()
    
    metrics = evaluator.evaluate_single_video(
        test_videos,
        tracker_config=eval_config.get('evaluation.tracker_config')
    )
    unlabeled_results[model_name] = metrics
    
    print(f"Results:")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

In [ ]:
# One for single video to start, just prints results in the notebook
# Then batch process all videos and aggregate

## Combine and Compare Results

In [ ]:
# Combine labeled and unlabeled results
all_results = {}
for model_name in MODELS.keys():
    all_results[model_name] = {
        **{f"labeled_{k}": v for k, v in labeled_results[model_name].items()},
        **{f"unlabeled_{k}": v for k, v in unlabeled_results[model_name].items()}
    }

comparison_df = pd.DataFrame(all_results).T

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(comparison_df.to_string())

# Save to CSV
csv_path = OUTPUT_DIR / "model_comparison.csv"
comparison_df.to_csv(csv_path)
print(f"\nComparison saved to: {csv_path}")

## Identify Best Model

In [ ]:
# Best model by labeled mAP50-95
if 'labeled_mAP50-95' in comparison_df.columns:
    best_labeled = comparison_df['labeled_mAP50-95'].idxmax()
    print(f"Best on labeled data (mAP50-95): {best_labeled}")
    print(f"  Score: {comparison_df.loc[best_labeled, 'labeled_mAP50-95']:.4f}")

# Best model by unlabeled avg confidence
if 'unlabeled_avg_confidence' in comparison_df.columns:
    best_confidence = comparison_df['unlabeled_avg_confidence'].idxmax()
    print(f"\nBest confidence on unlabeled: {best_confidence}")
    print(f"  Score: {comparison_df.loc[best_confidence, 'unlabeled_avg_confidence']:.4f}")

# Most consistent tracks
if 'unlabeled_avg_track_length' in comparison_df.columns:
    best_tracking = comparison_df['unlabeled_avg_track_length'].idxmax()
    print(f"\nBest tracking consistency: {best_tracking}")
    print(f"  Score: {comparison_df.loc[best_tracking, 'unlabeled_avg_track_length']:.4f}")